In [ ]:
import pandas as pd
import re
import os

# 1. Carregar com o cabeçalho correto (linha 5)
df_raw = pd.read_csv(
    "../data/raw/Vacinas_Aplicadas_Hercilio_Luz_Anonimizadas.csv",
    header=5
)

# Remover colunas extras não identificadas e linhas vazias
df_limpo = df_raw.dropna(how='all', axis=1)
df_limpo = df_limpo.dropna(subset=['Vacina, Estratégia e Dose']).copy()

# 2. Extração dos campos via Regex
# Padrão: "Nome da Vacina (Cod) - Estrategia (Cod) - Dose (Cod)"
padrao_vacina = r"^(.*?)\s*\((\d+)\)\s*-\s*(.*?)\s*\((\d+)\)\s*-\s*(.*?)\s*\((\d+)\)$"
extraidos = df_limpo['Vacina, Estratégia e Dose'].str.extract(padrao_vacina)

# 3. Separar Lote e Validade
# Padrão: "LOTE - DD/MM/AAAA"
padrao_lote = r"^(.*?)\s*-\s*(\d{2}/\d{2}/\d{4})$"
lote_extraido = df_limpo['Lote e Data de Validade'].str.extract(padrao_lote)

# 4. Montar o DataFrame no padrão exigido pela Issue
df_final = pd.DataFrame()
df_final['Nome do bairro'] = 'Hercilio Luz'
df_final['Nome da vacina'] = extraidos[0].str.strip()
df_final['Código da vacina'] = extraidos[1]
df_final['Estratégia'] = extraidos[2].str.strip()
df_final['Código da estratégia'] = extraidos[3]
df_final['Dose'] = extraidos[4].str.strip()
df_final['Código da dose'] = extraidos[5]
df_final['Vacina estratégia e dose'] = df_limpo['Vacina, Estratégia e Dose']
df_final['Lote'] = lote_extraido[0].str.strip()
df_final['Validade do lote'] = lote_extraido[1]
df_final['Data da aplicação'] = df_limpo['Aplicado']
df_final['Origem'] = df_limpo['Origem']

# Visualizar o resultado padronizado
df_final.head()

,Nome do bairro,Nome da vacina,Código da vacina,Estratégia,Código da estratégia,Dose,Código da dose,Vacina estratégia e dose,Lote,Validade do lote,Data da aplicação,Origem
0,NaN,Vacina meningo C,41,Rotina,1,2ª DOSE,2,Vacina meningo C (41) - Rotina (1) - 2ª DOSE (2),AMCB99AA,30/03/2027,20/02/2026,Tela de Aplicação
1,NaN,Vacina meningo C,41,Rotina,1,1ª DOSE,1,Vacina meningo C (41) - Rotina (1) - 1ª DOSE (1),AMCB99AA,30/03/2027,24/02/2026,Tela de Aplicação
2,NaN,Vacina meningo C,41,Rotina,1,2ª DOSE,2,Vacina meningo C (41) - Rotina (1) - 2ª DOSE (2),AMCC06AA,30/03/2027,22/04/2026,Prontuário
3,NaN,Vacina meningo C,41,Rotina,1,1ª DOSE,1,Vacina meningo C (41) - Rotina (1) - 1ª DOSE (1),AMCC06AA,30/03/2027,22/04/2026,Prontuário
4,NaN,Vacina meningo C,41,Rotina,1,REFORÇO,38,Vacina meningo C (41) - Rotina (1) - REFORÇO (38),AMCC06AA,30/03/2027,29/04/2026,Prontuário


In [14]:
# Montar o DataFrame no padrão exigido pela Issue
df_final = pd.DataFrame(index=df_limpo.index)
df_final['Nome do bairro'] = 'Hercilio Luz'
df_final['Nome da vacina'] = extraidos[0].str.strip()
df_final['Código da vacina'] = extraidos[1]
df_final['Estratégia'] = extraidos[2].str.strip()
df_final['Código da estratégia'] = extraidos[3]
df_final['Dose'] = extraidos[4].str.strip()
df_final['Código da dose'] = extraidos[5]
df_final['Vacina estratégia e dose'] = df_limpo['Vacina, Estratégia e Dose']
df_final['Lote'] = lote_extraido[0].str.strip()
df_final['Validade do lote'] = lote_extraido[1]
df_final['Data da aplicação'] = df_limpo['Aplicado']
df_final['Origem'] = df_limpo['Origem']

# Visualizar para confirmar o bairro preenchido
df_final.head()

,Nome do bairro,Nome da vacina,Código da vacina,Estratégia,Código da estratégia,Dose,Código da dose,Vacina estratégia e dose,Lote,Validade do lote,Data da aplicação,Origem
0,Hercilio Luz,Vacina meningo C,41,Rotina,1,2ª DOSE,2,Vacina meningo C (41) - Rotina (1) - 2ª DOSE (2),AMCB99AA,30/03/2027,20/02/2026,Tela de Aplicação
1,Hercilio Luz,Vacina meningo C,41,Rotina,1,1ª DOSE,1,Vacina meningo C (41) - Rotina (1) - 1ª DOSE (1),AMCB99AA,30/03/2027,24/02/2026,Tela de Aplicação
2,Hercilio Luz,Vacina meningo C,41,Rotina,1,2ª DOSE,2,Vacina meningo C (41) - Rotina (1) - 2ª DOSE (2),AMCC06AA,30/03/2027,22/04/2026,Prontuário
3,Hercilio Luz,Vacina meningo C,41,Rotina,1,1ª DOSE,1,Vacina meningo C (41) - Rotina (1) - 1ª DOSE (1),AMCC06AA,30/03/2027,22/04/2026,Prontuário
4,Hercilio Luz,Vacina meningo C,41,Rotina,1,REFORÇO,38,Vacina meningo C (41) - Rotina (1) - REFORÇO (38),AMCC06AA,30/03/2027,29/04/2026,Prontuário


In [15]:
# Criar a pasta de destino exigida pela issue
os.makedirs('../data/processed/vacinas', exist_ok=True)

# Salvar o arquivo padronizado
caminho_hercilio = '../data/processed/vacinas/hercilio_luz.csv'
df_final.to_csv(caminho_hercilio, index=False, sep=';', encoding='utf-8-sig')

print(f"Arquivo salvo com sucesso em: {caminho_hercilio}")

NameError: name 'os' is not defined